In [18]:
import Pkg; 
Pkg.add("ScenTrees"); Pkg.add("JLD2"); Pkg.add("GraphRecipes"); Pkg.add("Plots"); Pkg.add("DataFrames")
Pkg.add("HDF5")

using GraphRecipes, Plots, ScenTrees, JLD2
using DataFrames
using HDF5
include("functions.jl")

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
    Updating `~/.julia/environments/v1.11/Project.toml`
  [a93c6f00] + DataFrames v1.7.0
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [4]:
Case = "CCD22-3"

POIs=["WCASCADE", "JohnDay", "COTWDPGE", "Tesla", "Mossland"]
POI = POIs[1]

# cable length in meter
CabL = Dict("WCASCADE"=>545.060*1e3, "JohnDay"=>444.423*1e3, "COTWDPGE"=>324.193*1e3, "Tesla"=>603.598*1e3, "Mossland"=>660.732*1e3);

SolLoc = Case*"/"

KernTree = load(Case*"/ScenTree_"*POI*".jld2", "KernTree")

CabLength = CabL[POI];

N1 = 24; N2 = 4
T1 = [1:N1;]; # set of 24 hours in a day
T2 = [1:N1*N2;]; # set of 96 quarters in a day
T21 = Vector{Int64}[]; # set of 4 quarters in each hour
for t1 in T1
    push!(T21, [(t1-1)*N2+1:t1*N2;])
end

S1 = load(SolLoc*POI*"_Sol.jld2", "S1") # DA stage scenarios
S2 = load(SolLoc*POI*"_Sol.jld2", "S2") # RT stage scenarios
S21 = load(SolLoc*POI*"_Sol.jld2", "S21") # RT stage scenarios follow each DA stage scenario
pWS = load(SolLoc*POI*"_Sol.jld2", "pWS") # wind farm actual output T1, S2
pWDS = load(SolLoc*POI*"_Sol.jld2", "pWDS") # power traded in DA market T1, S1
pWRS = load(SolLoc*POI*"_Sol.jld2", "pWRS") # power traded in RT market T2, S2
pRWUS = load(SolLoc*POI*"_Sol.jld2", "pRWUS") # up reserve from wind farm T2, S2
pRWDS = load(SolLoc*POI*"_Sol.jld2", "pRWDS") # down reserve from wind farm T2, S2
pRBUS = load(SolLoc*POI*"_Sol.jld2", "pRBUS") # up reserve from battery T2, S2
pRBDS = load(SolLoc*POI*"_Sol.jld2", "pRBDS") # down reserve from battery T2, S2
ChS = load(SolLoc*POI*"_Sol.jld2", "ChS") # battery charged power T2, S2
DisS = load(SolLoc*POI*"_Sol.jld2", "DisS") # battery discharged power T2, S2
SCS = load(SolLoc*POI*"_Sol.jld2", "SCS") # battery state of charge
kWS = load(SolLoc*POI*"_Sol.jld2", "kWS") # control parameter for reserve from wind farm T2, S2
kBS = load(SolLoc*POI*"_Sol.jld2", "kBS") # control parameter for reserve from battery T2, S2
lam_DAQ = load(SolLoc*POI*"_Sol.jld2", "lam_DAQ") # DA price interpolated to T2
# DA price should be in T1 (hourly). I interpolated into T2 (every 15 minutes) for visualization
pWSQ = load(SolLoc*POI*"_Sol.jld2", "pWSQ") # wind farm actual output interpolated to T2
pWDSQ = load(SolLoc*POI*"_Sol.jld2", "pWDSQ") # power traded in DA market interpolated to T2
WPQ = load(SolLoc*POI*"_Sol.jld2", "WPQ") # wind farm generation capacity interpolated to T2
# results of passing wind speed to wind farm power curve
WSQ = load(SolLoc*POI*"_Sol.jld2", "WSQ") # wind speed interpolated to T2
lam_RT = load(SolLoc*POI*"_Sol.jld2", "lam_RT") # RT price T2, S2
PowerBase = load(SolLoc*POI*"_Sol.jld2", "PowerBase") # power base
Wrate = load(SolLoc*POI*"_Sol.jld2", "Wrate") # wind farm rated power

P1 = Dict() # probability of each DA stage scenario
P2 = Dict() # probability of each RT stage scenario
P21 = Dict() # probability of RT stage sceanrios for given DA stage scenario
for s in S1
    P1[s] = KernTree.probability[s]
end
for s in S2
    P2[s] = KernTree.probability[s]*KernTree.probability[KernTree.parent[s]]
    P21[s] = KernTree.probability[s]
end

States = Interpolate_States(KernTree);

lam_DA = Dict()
lam_RT = Dict()
lam_ReU = Dict()
lam_ReD = Dict()
WS = Dict()

for s in S1
    lam_DA[s] = KernTree.state[s, :] # DA price scenarios
end
for s in S2
    WS[s] = KernTree.state[s, :]
    col = KernTree.children[s+1][1]
    lam_RT[s] = States[col, :] # RT price scenarios
    col = KernTree.children[col+1][1]
    lam_ReU[s] = States[col, :] # Up reserve price scenarios
    col = KernTree.children[col+1][1]
    lam_ReD[s] = States[col, :] # Down reserve price scenarios
end
    
CabC = 2*0.3476*0.3351*1e6/525/1e3*CabLength # Cable material cost $/MW
CabFixC = 2*0.3476*0.0001*CabLength # Cable fixed cost
CabIntC = 2.7*1e6/16000*CabLength; # Cable installation cost
lam_ESS = 1.27009e6 # BESS cost $/MW, 4h duration

# other fixed cost
OnConv = 450*1e6 # onshore converter
OffConv = 772*1e6; # floating offshore converter

In [19]:
data = load(SolLoc*POI*"_Sol.jld2")

Dict{String, Any} with 26 entries:
  "v1"        => Dict{Any, Any}(56=>[510.0; 510.0; … ; 510.0; 510.0;;], 35=>[50…
  "pRBUS"     => Dict{Any, Any}(56=>[31.2388; 31.239; … ; 31.2199; 31.2077;;], …
  "pRWUS"     => Dict{Any, Any}(56=>[12.2171; 12.2171; … ; 12.2171; 12.2171;;],…
  "S1"        => [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 1…
  "pWDS"      => Dict{Any, Any}(5=>[900.68; 890.488; … ; 1080.61; 1050.37;;], 1…
  "kBS"       => Dict{Any, Any}(56=>[4.99822; 4.99825; … ; 4.99518; 4.99323;;],…
  "PowerBase" => 1500
  "SCS"       => Dict{Any, Any}(56=>[150.0; 150.0; … ; 0.00127862; 0.00120931;;…
  "pRBDS"     => Dict{Any, Any}(56=>[3.12388; 3.1239; … ; 3.12199; 3.12077;;], …
  "kWS"       => Dict{Any, Any}(56=>[2.00001; 2.00001; … ; 2.0; 2.0;;], 35=>[2.…
  "S2"        => [22, 23, 24, 25, 26, 27, 28, 29, 30, 31  …  112, 113, 114, 115…
  "S21"       => Dict{Any, Any}(5=>[37, 38, 39, 40, 41], 16=>[92, 93, 94, 95, 9…
  "lam_RT"    => Dict{Any, Any}(56=>[1.29747e5, 1.29

In [70]:
# flat_pairs = []
# for (k, mat) in data["v1"]
#     for i in 1:size(mat, 1)
#         push!(flat_pairs, Symbol("$(i)") => mat[i, 1])
#     end
# end
# df = DataFrame(flat_pairs)

   Resolving package versions...
   Installed libaec_jll ──────── v1.1.3+0
   Installed Hwloc_jll ───────── v2.12.0+0
   Installed HDF5_jll ────────── v1.14.6+0
   Installed OpenMPI_jll ─────── v5.0.7+2
   Installed MPIPreferences ──── v0.1.11
   Installed MicrosoftMPI_jll ── v10.1.4+3
   Installed MPICH_jll ───────── v4.3.0+1
   Installed HDF5 ────────────── v0.17.2
   Installed MPItrampoline_jll ─ v5.5.3+0
    Updating `~/.julia/environments/v1.11/Project.toml`
  [f67ccb44] + HDF5 v0.17.2
    Updating `~/.julia/environments/v1.11/Manifest.toml`
  [f67ccb44] + HDF5 v0.17.2
  [3da0fdf6] + MPIPreferences v0.1.11
  [0234f1f7] + HDF5_jll v1.14.6+0
  [e33a78d0] + Hwloc_jll v2.12.0+0
  [7cb0a576] + MPICH_jll v4.3.0+1
  [f1f71cc9] + MPItrampoline_jll v5.5.3+0
  [9237b28f] + MicrosoftMPI_jll v10.1.4+3
  [fe0851c0] + OpenMPI_jll v5.0.7+2
  [477f73a3] + libaec_jll v1.1.3+0
  [4af54fe1] + LazyArtifacts v1.11.0
Precompiling project...
    881.2 ms  ✓ MPIPreferences
    426.6 ms  ✓ libaec_jll
    

In [87]:
df = DataFrame([(Symbol(k) => vec(v)) for (k, v) in data["S1"]])
df

LoadError: BoundsError: attempt to access Int64 at index [2]

In [88]:
data["S1"]

20-element Vector{Int64}:
  2
  3
  4
  5
  6
  7
  8
  9
 10
 11
 12
 13
 14
 15
 16
 17
 18
 19
 20
 21

In [86]:
dfs = Dict{String, DataFrame}()
for (k, d) in data
    println(k)
    dfs[k] = DataFrame([(Symbol(k) => vec(v)) for (k, v) in d])
    # print(t3)
end


v1
pRBUS
pRWUS
S1


LoadError: BoundsError: attempt to access Int64 at index [2]

In [14]:
plot(KernTree, method=:tree, fontsize=10, nodeshape=:rect)

LoadError: Cannot convert Tree to series data for plotting

In [9]:
DCFR = sum(1/(1+0.03)^i for i=0:15) # discount cash flow rate
DAR = sum(lam_DA[s][t]*pWDS[s][t]*P1[s] for t in T1, s in S1)/1e3 # DA revenue $k
RTR = sum(lam_RT[s][t]*pWRS[s][t]*P2[s] for t in T2, s in S2)/4/1e3 # RT revenue $k
REUR = sum(lam_ReU[s][t]*(pRWUS[s][t]+pRBUS[s][t])*P2[s] for t in T2, s in S2)/4/1e3 # up reserve revenue $k
REDR = sum(lam_ReD[s][t]*(pRWDS[s][t]+pRBDS[s][t])*P2[s] for t in T2, s in S2)/4/1e3 # down reserve revenue $k
LR = DCFR*365*(DAR+RTR+REUR+REDR)/1e3 # life time revenue $M
;